# SOLiD-PINN v0.8.0 — Open benchmark inventory
Select CPU and Run all. Authorize Drive. Downloaded files are stored privately. All three open experimental/metadata sources are enabled. No training, model fitting or unit assumptions. A parsed file is not yet a model-ready dataset.

In [ ]:
from google.colab import drive
from pathlib import Path
import sys, subprocess, json
drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/SOLiD-PINN/open_benchmarks')
CACHE=ROOT/'downloads'
REPORT=ROOT/'inventory.json'
REPO=Path('/content/SOLiD-PINN-v080')
REV="87349feee07ccd6c36b126f36d54375fa062ee87"
SOURCES=["umd","puls2024","liopa"]
if not (REPO/".git").exists():
    subprocess.run(["git","clone","--no-checkout","https://github.com/theerayutbo/SOLiD-PINN.git",str(REPO)],check=True)
subprocess.run(["git","fetch","origin",REV],cwd=REPO,check=True)
subprocess.run(["git","checkout","--detach",REV],cwd=REPO,check=True)
assert subprocess.check_output(["git","rev-parse","HEAD"],cwd=REPO,text=True).strip()==REV
subprocess.run([sys.executable,"-m","pip","install","openpyxl==3.1.5","pytest"],check=True)
subprocess.run([sys.executable,"-m","pytest","-q","tests/test_open_benchmark_audit.py"],cwd=REPO,check=True)
SETUP_OK=True

## Acquire and inventory
If a file is blocked, the report still records its error. Download it manually from the publisher and place it under CACHE/source_id/file_name, then rerun. Do not overwrite a verified original.

In [ ]:
assert SETUP_OK
REPORT.unlink(missing_ok=True)
cmd=[sys.executable,"scripts/audit_open_benchmarks.py","--cache",str(CACHE),"--out",str(REPORT),"--download"]
for source in SOURCES: cmd+=["--source",source]
p=subprocess.run(cmd,cwd=REPO)
if p.returncode not in (0,1) or not REPORT.is_file(): raise RuntimeError("Inventory command failed; inspect logs")
result=json.loads(REPORT.read_text())
for item in result["files"]:
    print(item["source"],item["file"],item["status"],item.get("error",""))
    profile=item.get("profile",{})
    if "rows" in profile: print("  rows:",profile["rows"],"columns:",profile["columns"],"malformed:",profile["malformed_rows"])
    for sheet in profile.get("sheets",[]): print("  sheet:",sheet["name"],"declared dimensions:",sheet["declared_rows"],sheet["declared_columns"])
print("Inventory:",REPORT)
if any(item["status"]!="inventoried" for item in result["files"]):
    print("PARTIAL/BLOCKED: not all files available or parsed.")

## Export inventory for review
Download metadata, field names, counts, hashes and errors only. Raw curves remain in Drive. Confirm axes, units, current, cycle ID, cell IDs and chemistry-specific model requirements before ingestion.

In [ ]:
from google.colab import files
files.download(str(REPORT))